In [1]:
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(0)

b=2
k=3
t=2
X = torch.rand(b,t,k)
print(f'The shape of X is {X.shape}')
raw_weights = torch.bmm(X, X.transpose(1, 2))
print(raw_weights.shape)
print(raw_weights)

The shape of X is torch.Size([2, 2, 3])
torch.Size([2, 2, 2])
tensor([[[0.8443, 0.3578],
         [0.3578, 0.5140]],

        [[1.2514, 0.8057],
         [0.8057, 0.6829]]])


In [19]:
weights = F.softmax(raw_weights, dim=2)
print(weights)

tensor([[[0.6193, 0.3807],
         [0.4610, 0.5390]],

        [[0.6096, 0.3904],
         [0.5307, 0.4693]]])


In [21]:
y = torch.bmm(weights, X)
print(y)

tensor([[[0.3576, 0.5928, 0.2962],
         [0.2999, 0.5199, 0.3825]],

        [[0.5456, 0.6827, 0.4346],
         [0.5568, 0.6395, 0.4303]]])


In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, k, heads=4, mask=False):
        super().__init__()
        
        assert k % heads == 0, "Embedding size must be divisible by number of heads"

        self.tokeys = nn.Linear(k, k, bias=False)
        self.toqueries = nn.Linear(k, k, bias=False)
        self.tovalues = nn.Linear(k, k, bias=False)

        self.unifyheads = nn.Linear(k, k)
    
    def forward(self, x):
        b, t, k = x.size()
        h = self.heads

        queries = self.toqueries(x)
        keys = self.tokeys(x)
        values= self.tovalues(x)

        s = k//h
        keys = keys.view(b, t, h, s)
        queries = queries.view(b, t, h, s)
        values = values.view(b, t, h, s)

        # fold heads into the batch dimension
        keys = keys.transpose(1, 2).contiguous().view(b*h, t, s)
        queries = queries.transpose(1, 2).contiguous().view(b*h, t, s)
        values = values.transpose(1, 2).contiguous().view(b*h, t, s)

        # dot product of queries and keys
        dot = torch.bmm(queries, keys.transpose(1, 2))

        # scale the dot
        dot = dot / (s ** 0.5)

        #normalize
        dot = F.softmax(dot,dim=2)

        # apply the self attention to the values
        out = torch.bmm(dot, values).view(b, h, t, s)

        # swap h, t back, unify heads 
        # 因为tensor shape的限制，这里我们需要两部操作才能调成正确的顺序
        out = out.transpose(1,2).contiguous().view(b,t,s*h)


In [ ]:
class TransformerBlock(nn.Module):

    def __init__(self, k, heads):
        '''
        k: int: number of input embeddings
        heads: int: number of attention heads
        '''
        super().__init__()

        self.attention = SelfAttention(k, heads=heads)

        self.norm1 = nn.LayerNorm(k)
        self.norm2 = nn.LayerNorm(k)

        # the mlp block
        self.ff = nn.Sequential(
            nn.Linear(k, heads*k),
            nn.ReLU(),
            nn.Linear(heads*k, k)
        )

    def forward(self, x):
        attended = self.attention(x)
        x = self.norm1(attended + x)

        fedforward = self.ff(x)
        return self.norm2(fedforward+x)